In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin


In [2]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, size_ix, lat_ix, lng_ix = 0, 1, 4, 5

class CustomFeaturesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, center_lat=41.3171, center_lng=69.2797):
        # Initialize center coordinates
        self.center_lat = center_lat
        self.center_lng = center_lng
        
    def fit(self, X, y=None):
        return self # Transformer only transforms, does not learn anything
        
    def transform(self, X):
        X = np.asarray(X)
        
        # 1. Calculate room_size (size / rooms)
        room_size = X[:, size_ix] / X[:, rooms_ix]
        
        # 2. Calculate distance_to_center (Haversine)
        R = 6371 # Earth radius
        
        # Take lat and lng columns from X array and convert to radians
        lat1, lon1 = map(np.radians, [X[:, lat_ix], X[:, lng_ix]])
        lat2, lon2 = map(np.radians, [self.center_lat, self.center_lng])
        
        dlon = lon2 - lon1 
        dlat = lat2 - lat1 
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a)) 
        distance_to_center = R * c
        
        # Add two new columns to the original X array (using np.c_)
        return np.c_[X, room_size, distance_to_center]

In [3]:
df = pd.read_csv("../data/processed/tashkent_houses.csv")
df.head()

,district,rooms,size,level,max_levels,price,lat,lng
0,Yunusobod,3,57.0,4,4,52000,41.371471,69.281049
1,Yakkasaroy,2,52.0,4,5,56000,41.291115,69.261104
2,Chilonzor,2,42.0,4,4,37000,41.280784,69.223683
3,Chilonzor,3,65.0,1,4,49500,41.290163,69.196862
4,Chilonzor,3,70.0,3,5,55000,41.300156,69.210831


In [4]:
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=df['district'])
X_train_num = X_train.drop('district', axis=1)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(5924, 7) (5924,)
(1481, 7) (1481,)


In [5]:
numeric_attr =  list(X_train_num)
categorical_attr = ['district']

In [6]:
num_pipeline = Pipeline([
    ('custom_features_adder', CustomFeaturesAdder()),
    ('std_scaler', StandardScaler()) 
])

num_pipeline.fit_transform(X_train_num.values)[:,1]

array([ 1.35526899, -0.07444581,  0.26868574, ..., -0.78930321,
       -0.36038877, -0.70352032], shape=(5924,))

In [7]:
full_pipeline = ColumnTransformer([
    ('num', num_pipeline, numeric_attr),
    ('cat', OneHotEncoder(), categorical_attr)
])

In [8]:
X_train_prepared = full_pipeline.fit_transform(X_train)
X_train_prepared[:1]

array([[ 1.30661963,  1.35526899,  0.1400949 ,  0.7584311 ,  0.81949687,
         0.88977987,  0.18023521, -1.00570271,  0.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ]])

In [9]:
#LinerRegression
from sklearn.linear_model import LinearRegression

LR_model = LinearRegression()
LR_model.fit(X_train_prepared, y_train)

print('Configurations! Your model is trained!')

Configurations! Your model is trained!


In [10]:
test_data = X_test.sample(5)
test_data

,district,rooms,size,level,max_levels,lat,lng
6592,Chilonzor,1,42.0,3,9,41.259418,69.180471
4419,Mirzo Ulugbek,2,59.0,4,8,41.313367,69.323575
3602,Mirobod,2,56.0,3,5,41.245914,69.307123
6147,Shayhontohur,2,57.0,4,5,41.334551,69.167805
2692,Yakkasaroy,4,150.0,7,7,41.298686,69.259508


In [11]:
test_labels = y_test.loc[test_data.index]
test_labels

6592     26500
4419     42000
3602     38500
6147     35000
2692    112000
Name: price, dtype: int64

In [12]:
test_data_prepared = full_pipeline.transform(test_data)

In [13]:
predict_test_data = LR_model.predict(test_data_prepared)
predict_test_data

array([ 26325.12752868,  53717.64991606,  43042.36473622,  34570.51063896,
       139150.23552742])

In [14]:
pd.DataFrame({
    'Bashorat': predict_test_data.round(1),
    'Asil narx': test_labels.values,
    'Farq': abs(predict_test_data - test_labels.values).round(1)
})

,Bashorat,Asil narx,Farq
0,26325.1,26500,174.9
1,53717.6,42000,11717.6
2,43042.4,38500,4542.4
3,34570.5,35000,429.5
4,139150.2,112000,27150.2


In [15]:
X_test_prepared = full_pipeline.transform(X_test)
X_test_prepared[:1]

array([[ 0.36702086, -0.21741729, -1.2028213 , -0.76705768, -0.96810997,
        -0.78871517, -0.90085016,  0.60256514,  0.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ]])

In [16]:
LR_model_predict = LR_model.predict(X_test_prepared)
LR_model_predict

array([50058.12557669, 26293.28338335, 58010.53228532, ...,
       51141.49645646, 74105.80903472, 54644.09022679], shape=(1481,))

In [17]:
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_test, LR_model_predict)
print(f"Test RMSE: {rmse:.2f}")

Test RMSE: 19492.90


In [18]:
#RandomForestRegressor

from sklearn.ensemble import RandomForestRegressor
RF_model = RandomForestRegressor(n_estimators=100, random_state=42)
RF_model.fit(X_train_prepared, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [19]:
RF_model_predict = RF_model.predict(X_test_prepared)
RF_model_predict


array([47831.07      , 27540.09666667, 59603.94666667, ...,
       48019.38095238, 74179.85      , 51640.83333333], shape=(1481,))

In [20]:
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_test, RF_model_predict)
print(f"Test RMSE: {rmse:.2f}")

Test RMSE: 16200.90


In [21]:
#TreeRegressor

from sklearn.tree import DecisionTreeRegressor

DT_model = DecisionTreeRegressor(random_state=42)   

DT_model.fit(X_train_prepared, y_train) 

DT_model_predict = DT_model.predict(X_test_prepared)
DT_model_predict


array([45500., 26500., 75000., ..., 46500., 52000., 68000.], shape=(1481,))

In [22]:
from sklearn.metrics import root_mean_squared_error
rmse_DT = root_mean_squared_error(y_test, DT_model_predict)
print(f"Test RMSE: {rmse_DT:.2f}")

Test RMSE: 23639.42


In [23]:
#Cross validation 
from sklearn.model_selection import cross_val_score

def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Std.dev:", scores.std())

X_cross_val_prepared = full_pipeline.transform(X)

In [24]:
val_score_RF = cross_val_score(RF_model, X_cross_val_prepared, y, scoring='neg_root_mean_squared_error', cv=10)
val_score_RF

array([-14422.09178274, -14470.64798165, -19554.33116138, -21460.45273031,
       -19526.23889486, -23726.72782951, -19882.22008401, -23574.69558192,
       -17319.50611743, -12924.32264317])

In [25]:
display_scores(-val_score_RF)

Scores: [14422.09178274 14470.64798165 19554.33116138 21460.45273031
 19526.23889486 23726.72782951 19882.22008401 23574.69558192
 17319.50611743 12924.32264317]
Mean: 18686.123480698603
Std.dev: 3618.164585239206


In [26]:
val_score_LR = cross_val_score(LR_model, X_cross_val_prepared, y, scoring='neg_root_mean_squared_error', cv=7)
val_score_LR

array([-18584.42406441, -22385.29724128, -24364.37601245, -26344.65611608,
       -24255.47938319, -21637.6867441 , -20141.12144672])

In [27]:
display_scores(-val_score_LR)

Scores: [18584.42406441 22385.29724128 24364.37601245 26344.65611608
 24255.47938319 21637.6867441  20141.12144672]
Mean: 22530.434429748824
Std.dev: 2478.0547226908675


In [28]:
val_score_DT = cross_val_score(DT_model, X_cross_val_prepared, y, scoring='neg_root_mean_squared_error', cv=7)
val_score_DT

array([-19317.54461129, -24622.27366369, -27142.00614991, -24696.8286282 ,
       -24294.50408946, -25162.83479201, -23810.60055656])

In [29]:
display_scores(-val_score_DT)

Scores: [19317.54461129 24622.27366369 27142.00614991 24696.8286282
 24294.50408946 25162.83479201 23810.60055656]
Mean: 24149.513213018374
Std.dev: 2203.526262227116


In [30]:
import joblib
joblib.dump(RF_model, '../models/random_forest_model.jbl')

['../models/random_forest_model.jbl']

In [31]:
joblib.dump(LR_model, '../models/logistic_regression_model.jbl')

['../models/logistic_regression_model.jbl']

In [32]:
import pickle
pickle.dump(DT_model, open('../models/decision_tree_model.pkl', 'wb'))
